# MVP de Engenharia de Dados
## Notebook 2 — Coleta dos Dados do ONS

### Objetivo deste notebook

Este notebook é responsável pela etapa de coleta dos dados utilizados no MVP.

A fonte selecionada é o portal de Dados Abertos do Operador Nacional do Sistema Elétrico (ONS), utilizando o conjunto de dados **Balanço de Energia nos Subsistemas**.

O conjunto contém informações de carga e oferta de energia em periodicidade horária por subsistema. A oferta de energia contempla valores de geração hidráulica, térmica, eólica e fotovoltaica, medidos em MWmed.

Para este projeto serão utilizados os dados correspondentes aos anos de **2020 a 2025**, permitindo analisar seis anos completos e comparar o comportamento da carga e das diferentes fontes de geração ao longo do tempo.

### Formato selecionado

Os dados serão coletados no formato **Parquet**, formato colunar adequado ao processamento analítico e suportado pelo Apache Spark utilizado pelo Databricks.

### Licença e utilização dos dados

Os dados são disponibilizados pelo ONS sob licença **Creative Commons Atribuição (CC-BY)**. Essa licença permite a reutilização, distribuição e adaptação dos dados, desde que seja atribuído o devido crédito ao ONS e sejam informadas eventuais alterações realizadas.

**Fonte:** Operador Nacional do Sistema Elétrico (ONS)  
**Conjunto de dados:** Balanço de Energia nos Subsistemas  
**Período utilizado:** 2020 a 2025

In [0]:
import requests

url_2020 = "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/balanco_energia_subsistema_ho/BALANCO_ENERGIA_SUBSISTEMA_2020.parquet"

try:
    resposta = requests.get(url_2020, timeout=60)

    print("Status HTTP:", resposta.status_code)
    print("Tamanho do arquivo:", len(resposta.content), "bytes")

except requests.RequestException as erro:
    resposta = None

    print("Fonte do ONS temporariamente indisponível para acesso pelo ambiente Databricks.")
    print("Erro de conexão:", erro)

Fonte do ONS temporariamente indisponível para acesso pelo ambiente Databricks.
Erro de conexão: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


In [0]:
import os

print("Diretório atual:", os.getcwd())

Diretório atual: /Workspace/Users/irvinglui83@gmail.com


In [0]:
caminho_2020 = os.path.join(
    os.getcwd(),
    "BALANCO_ENERGIA_SUBSISTEMA_2020.parquet"
)

if resposta is not None:
    with open(caminho_2020, "wb") as arquivo:
        arquivo.write(resposta.content)

    print("Arquivo 2020 salvo a partir da fonte do ONS.")

else:
    pasta_download = os.path.join(os.getcwd(), "dados_ons")
    caminho_2020 = os.path.join(
        pasta_download,
        "BALANCO_ENERGIA_SUBSISTEMA_2020.parquet"
    )

    print("Download não realizado nesta execução.")
    print("Utilizando o arquivo 2020 coletado anteriormente.")

print("Arquivo localizado em:", caminho_2020)
print("Arquivo existe:", os.path.exists(caminho_2020))
print("Tamanho:", os.path.getsize(caminho_2020), "bytes")

Download não realizado nesta execução.
Utilizando o arquivo 2020 coletado anteriormente.
Arquivo localizado em: /Workspace/Users/irvinglui83@gmail.com/dados_ons/BALANCO_ENERGIA_SUBSISTEMA_2020.parquet
Arquivo existe: True
Tamanho: 2555477 bytes


In [0]:
df_2020 = spark.read.parquet(f"file:{caminho_2020}")

display(df_2020.limit(10))

id_subsistema,nom_subsistema,din_instante,val_gerhidraulica,val_gertermica,val_gereolica,val_gersolar,val_carga,val_intercambio
NE,NORDESTE,2020-01-01T00:00:00.000Z,2226.54000000,2161.19500000,1852.16399999,0E-8,9617.97500000,-3378.07600000
N,NORTE,2020-01-01T00:00:00.000Z,5309.50200000,2891.61400000,23.79600000,0E-8,4889.14600000,3335.76600000
SIN,SISTEMA INTERLIGADO NACIONAL,2020-01-01T00:00:00.000Z,41115.41300000,14355.63950000,2088.36400000,0E-8,57629.72000000,-70.31000000
SE,SUDESTE/CENTRO-OESTE,2020-01-01T00:00:00.000Z,31443.04200000,7647.70550000,0E-8,0E-8,33108.86150000,5981.88600000
S,SUL,2020-01-01T00:00:00.000Z,2136.32900000,1655.12499999,212.40400000,0E-8,10013.74200000,-6009.88400000
NE,NORDESTE,2020-01-01T01:00:00.000Z,2574.69600000,2164.20200000,1520.24600000,0E-8,9628.51700000,-3369.37300000
N,NORTE,2020-01-01T01:00:00.000Z,5244.15799999,2895.89300000,14.63600000,0E-8,4953.31499999,3201.37200000
SIN,SISTEMA INTERLIGADO NACIONAL,2020-01-01T01:00:00.000Z,42552.90900000,13967.38550000,1732.24700000,0E-8,58322.84000000,-70.30000000
SE,SUDESTE/CENTRO-OESTE,2020-01-01T01:00:00.000Z,32644.45000000,7253.79850000,0E-8,0E-8,33368.40850000,6529.84000000
S,SUL,2020-01-01T01:00:00.000Z,2089.60500000,1653.49200000,197.36500000,0E-8,10372.60399999,-6432.14200000


In [0]:
df_2020.printSchema()

root
 |-- id_subsistema: string (nullable = true)
 |-- nom_subsistema: string (nullable = true)
 |-- din_instante: timestamp (nullable = true)
 |-- val_gerhidraulica: string (nullable = true)
 |-- val_gertermica: string (nullable = true)
 |-- val_gereolica: string (nullable = true)
 |-- val_gersolar: string (nullable = true)
 |-- val_carga: string (nullable = true)
 |-- val_intercambio: string (nullable = true)



In [0]:
quantidade_2020 = df_2020.count()

print("Quantidade de registros em 2020:", quantidade_2020)

Quantidade de registros em 2020: 43920


### Validação inicial da coleta

Antes de automatizar a coleta de todo o período, foi realizado um teste utilizando o arquivo referente ao ano de 2020.

O acesso ao recurso foi validado por meio de requisição HTTP, com retorno de status `200`, indicando que o arquivo estava disponível para download.

Após o armazenamento do arquivo no ambiente do Databricks, o conteúdo foi lido com Apache Spark e sua estrutura foi inspecionada.

O arquivo de 2020 apresentou **43.920 registros** e 9 atributos:

- `id_subsistema`
- `nom_subsistema`
- `din_instante`
- `val_gerhidraulica`
- `val_gertermica`
- `val_gereolica`
- `val_gersolar`
- `val_carga`
- `val_intercambio`

A coluna `din_instante` foi reconhecida como `timestamp`, enquanto os atributos relacionados aos valores de geração, carga e intercâmbio foram identificados como `string`.

Neste momento não serão realizadas conversões ou tratamentos nesses atributos, pois a padronização dos tipos de dados será realizada posteriormente na camada Silver. A etapa de coleta tem como objetivo preservar os arquivos obtidos da fonte antes das transformações do pipeline.

In [0]:
anos = range(2020, 2026)

url_base = (
    "https://ons-aws-prod-opendata.s3.amazonaws.com/"
    "dataset/balanco_energia_subsistema_ho/"
    "BALANCO_ENERGIA_SUBSISTEMA_{}.parquet"
)

for ano in anos:
    url = url_base.format(ano)
    print(ano, "->", url)

2020 -> https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/balanco_energia_subsistema_ho/BALANCO_ENERGIA_SUBSISTEMA_2020.parquet
2021 -> https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/balanco_energia_subsistema_ho/BALANCO_ENERGIA_SUBSISTEMA_2021.parquet
2022 -> https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/balanco_energia_subsistema_ho/BALANCO_ENERGIA_SUBSISTEMA_2022.parquet
2023 -> https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/balanco_energia_subsistema_ho/BALANCO_ENERGIA_SUBSISTEMA_2023.parquet
2024 -> https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/balanco_energia_subsistema_ho/BALANCO_ENERGIA_SUBSISTEMA_2024.parquet
2025 -> https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/balanco_energia_subsistema_ho/BALANCO_ENERGIA_SUBSISTEMA_2025.parquet


In [0]:
import os
import requests

pasta_download = os.path.join(os.getcwd(), "dados_ons")

os.makedirs(pasta_download, exist_ok=True)

for ano in anos:
    url = url_base.format(ano)
    nome_arquivo = f"BALANCO_ENERGIA_SUBSISTEMA_{ano}.parquet"
    caminho_arquivo = os.path.join(pasta_download, nome_arquivo)

    try:
        resposta = requests.get(url, timeout=60)

        if resposta.status_code == 200:
            with open(caminho_arquivo, "wb") as arquivo:
                arquivo.write(resposta.content)

            print(
                f"{ano} | HTTP {resposta.status_code} | "
                f"{len(resposta.content):,} bytes | OK"
            )
        else:
            print(f"{ano} | HTTP {resposta.status_code} | ERRO")

    except requests.RequestException:
        if os.path.exists(caminho_arquivo):
            print(
                f"{ano} | fonte temporariamente indisponível | "
                f"arquivo já coletado disponível | OK"
            )
        else:
            print(
                f"{ano} | fonte temporariamente indisponível | "
                f"arquivo não encontrado | ERRO"
            )

2020 | fonte temporariamente indisponível | arquivo já coletado disponível | OK
2021 | fonte temporariamente indisponível | arquivo já coletado disponível | OK
2022 | fonte temporariamente indisponível | arquivo já coletado disponível | OK
2023 | fonte temporariamente indisponível | arquivo já coletado disponível | OK
2024 | fonte temporariamente indisponível | arquivo já coletado disponível | OK
2025 | fonte temporariamente indisponível | arquivo já coletado disponível | OK


In [0]:
arquivos_baixados = sorted(os.listdir(pasta_download))

print("Quantidade de arquivos:", len(arquivos_baixados))
print()

for arquivo in arquivos_baixados:
    caminho = os.path.join(pasta_download, arquivo)
    tamanho = os.path.getsize(caminho)

    print(f"{arquivo} | {tamanho:,} bytes")

Quantidade de arquivos: 6

BALANCO_ENERGIA_SUBSISTEMA_2020.parquet | 2,555,477 bytes
BALANCO_ENERGIA_SUBSISTEMA_2021.parquet | 2,623,319 bytes
BALANCO_ENERGIA_SUBSISTEMA_2022.parquet | 2,750,811 bytes
BALANCO_ENERGIA_SUBSISTEMA_2023.parquet | 2,075,363 bytes
BALANCO_ENERGIA_SUBSISTEMA_2024.parquet | 2,086,316 bytes
BALANCO_ENERGIA_SUBSISTEMA_2025.parquet | 2,105,644 bytes


In [0]:
total_registros = 0

for ano in range(2020, 2026):
    caminho_ano = f"file:{pasta_download}/BALANCO_ENERGIA_SUBSISTEMA_{ano}.parquet"

    df_ano = spark.read.parquet(caminho_ano)

    quantidade = df_ano.count()
    total_registros += quantidade

    print(f"{ano}: {quantidade:,} registros")

print()
print(f"Total de registros de 2020 a 2025: {total_registros:,}")

2020: 43,920 registros
2021: 43,800 registros
2022: 43,800 registros
2023: 43,800 registros
2024: 43,920 registros
2025: 43,800 registros

Total de registros de 2020 a 2025: 263,040


## Resultado da coleta

A coleta dos dados do ONS foi realizada para o período de **2020 a 2025**.

Foram obtidos seis arquivos no formato Parquet, correspondentes a um arquivo para cada ano analisado. Durante a coleta original, o retorno HTTP dos recursos foi validado, sendo obtido status `200` para os arquivos utilizados no projeto.

Na revisão final do pipeline, o acesso ao endpoint do ONS a partir do ambiente Databricks apresentou indisponibilidade temporária. Como os arquivos coletados anteriormente permaneciam armazenados no ambiente, o pipeline foi preparado para reutilizá-los quando a fonte externa não estivesse acessível, preservando a continuidade do processamento.

Os arquivos foram validados individualmente com Apache Spark, permitindo verificar sua leitura e a quantidade de registros de cada período.

A quantidade de registros identificada por ano foi:

- **2020:** 43.920 registros
- **2021:** 43.800 registros
- **2022:** 43.800 registros
- **2023:** 43.800 registros
- **2024:** 43.920 registros
- **2025:** 43.800 registros

O conjunto completo apresentou **263.040 registros**.

A diferença observada em 2020 e 2024 é compatível com o fato de serem anos bissextos. Considerando a estrutura observada de cinco registros por hora, os anos com 366 dias apresentam 43.920 registros, enquanto os anos com 365 dias apresentam 43.800 registros.

Essas contagens serão utilizadas como referência nas etapas seguintes do pipeline, permitindo verificar se a quantidade de registros é preservada durante a ingestão e as transformações.

Nesta etapa não foram realizadas transformações nos dados coletados. Os arquivos foram mantidos no formato Parquet disponibilizado pela fonte para posterior ingestão na camada Bronze.
